# Quantitative Study 02: Expiry-Day 0DTE Greeks & Taylor P&L Attribution
**Author**: Quantitative Research Desk  
**Objective**: Study 0DTE gamma spikes on weekly expiry days, evaluate ROC leg cutting vs static SL, and decompose intraday P&L into mathematical Greeks.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from options_bt.backtester import OptionsBacktester
from options_bt.leg_cutter import LegCutConfig
from options_bt.pricing import Black76

### 1. Execute 0DTE Expiry Simulation (January 2, 2025 Thursday Expiry)

In [ ]:
spot_path = "../sample_data/spot/nifty50_1min_sample.csv"
spot_df = pd.read_csv(spot_path)
spot_df["date"] = pd.to_datetime(spot_df["date"])

options_file = "../sample_data/options/NIFTY_20250102.csv"

# Configure Leg Cutter with dual barriers
leg_cfg = LegCutConfig(
    profit_target_points=25.0,
    static_sl_points=35.0,
    roc_window=5,
    roc_threshold_pct=0.35
)

bt = OptionsBacktester(lot_size=25, num_lots=4, leg_config=leg_cfg)
result = bt.run_session(options_file, spot_df, "2025-01-02", entry_time="09:25:00")

print(f"--- 0DTE Expiry Session Results ({result.date}) ---")
print(f"ATM Strike: {result.atm_strike} | Anchor Spot: {result.entry_spot:.2f}")
print(f"Straddle Entry: ₹{result.straddle_entry:.2f} (CE: {result.ce_entry:.2f}, PE: {result.pe_entry:.2f})")
print(f"CE Exit: ₹{result.ce_exit:.2f} ({result.ce_reason})")
print(f"PE Exit: ₹{result.pe_exit:.2f} ({result.pe_reason})")
print(f"Gross Points: {result.gross_points:+.2f} | Net Points: {result.net_points:+.2f}")
print(f"Gross P&L: ₹{result.gross_pnl_rupees:+,.2f} | Net P&L: ₹{result.net_pnl_rupees:+,.2f}")
print(f"Statutory Costs: ₹{result.statutory_costs_rupees:,.2f}")

### 2. Greek P&L Attribution Decomposition

In [ ]:
attr = result.attributions
attr_df = pd.DataFrame(list(attr.items()), columns=["Component", "PnL_Rupees"])
print(attr_df)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(attr_df["Component"], attr_df["PnL_Rupees"], color=["green" if x>=0 else "red" for x in attr_df["PnL_Rupees"]])
ax.set_title("0DTE Session P&L Attribution (Taylor Series Decomposition)")
ax.set_ylabel("P&L (₹)")
plt.xticks(rotation=30)
plt.show()